## Explaining Anomalies with SHAP using Isolation Forest

- `Isolation Forest` has become a quick staple in anomaly detection because of its ability to find complex anomalies in large datasets with many features.

- `Anomaly detection` involves identifying unusual instances that are not consistent with the rest of the data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.ensemble import IsolationForest
from ucimlrepo import fetch_ucirepo

shap.initjs()
plt.rcParams.update({"figure.facecolor": "white"})

In [ ]:
# Fetch dataset from UCI repository
power_consumption=fetch_ucirepo(id=235)

In [ ]:
print(power_consumption.variables)

In [ ]:
# Get all the features
data=power_consumption.data.features
data['Date']=pd.to_datetime(data['Date'], format='%d/%m/%Y')

# List of features to check
feature_columns=['Global_active_power', 'Global_reactive_power',
                 'Voltage', 'Global_intensity', 'Sub_metering_1',
                 'Sub_metering_2', 'Sub_metering_3']

# Convert columns to numeric and replace any errors with NaN
data[feature_columns]=data[feature_columns].apply(
    pd.to_numeric,
    errors='coerce'
)

# Drop rows where all feature columns are missing(NaN)
data_cleaned=data.dropna(subset=feature_columns, how='all')

data_cleaned.head()

In [ ]:
data_cleaned.info()

In [ ]:
# Group by Date and calculate mean and std deviation
data_aggregated=data_cleaned.groupby('Date')[feature_columns].agg(['mean', 'std'])

# Rename columns to the desired format
data_aggregated.columns=[
    f'{agg_type.upper()}_{col}' for col, agg_type in data_aggregated.columns
]

# Reset the index
data_aggregated.reset_index(inplace=True)

In [ ]:
data_aggregated.shape

In [ ]:
data_aggregated.head()

### Training the Isolation Forest

In [ ]:
# Number of trees
n_estimators=100
# Number of samples used to train each tree
sample_size=256
# Expected proportion of anomalies
contamination=0.02

In [ ]:
features=data_aggregated.drop(columns='Date', axis=1)

iso_forest=IsolationForest(
    n_estimators=n_estimators,
    contamination=contamination,
    max_samples=sample_size,
    random_state=42
)

In [ ]:
iso_forest.fit(features)

In [ ]:
data_aggregated['anomaly_score']=iso_forest.decision_function(features)
data_aggregated['anomaly']=iso_forest.predict(features)

In [ ]:
data_aggregated['anomaly'].value_counts()

In [ ]:
plt.figure(figsize=(10, 5))

# Plotting normal instances
normal=data_aggregated[data_aggregated['anomaly']==1]
plt.scatter(normal['Date'], normal['anomaly_score'], label='Normal')

# Plotting anomaly instances
anomaly=data_aggregated[data_aggregated['anomaly']==-1]
plt.scatter(anomaly['Date'], anomaly['anomaly_score'], label='Anomaly')

plt.xlabel('Instances')
plt.ylabel('Anomaly Score')
plt.legend()

### KernelSHAP with Anomaly Score

In [ ]:
# Select all anomalies and 100 random normal samples
normal_sample=np.random.choice(normal.index, size=100, replace=False)
sample=np.append(anomaly.index, normal_sample)

In [ ]:
print(len(sample))

In [ ]:
explainer=shap.Explainer(iso_forest.decision_function, features)
shap_values=explainer(features.iloc[sample])

In [ ]:
shap.plots.waterfall(shap_values[0])

In [ ]:
shap.plots.waterfall(shap_values[100])

In [ ]:
shap.plots.bar(shap_values)

In [ ]:
shap.plots.beeswarm(shap_values)

### TreeSHAP with Average Path Length

- TreeSHAP is faster than KernelSHAP and wecan even use the shap interaction values.

In [ ]:
tree_explainer=shap.TreeExplainer(iso_forest)
shap_values_tree=tree_explainer(features)

In [ ]:
shap.plots.waterfall(shap_values_tree[0])

In [ ]:
path_length=shap_values_tree.base_values+shap_values_tree.values.sum(axis=1)

anomalies=data_aggregated[data_aggregated['anomaly']==-1]
path_length_anomaly=path_length[anomalies.index]

normal=data_aggregated[data_aggregated['anomaly']==1]
path_length_normal=path_length[normal.index]

plt.figure(figsize=(10, 5))
plt.boxplot([path_length_anomaly, path_length_normal], labels=['Anomaly', 'Normal'])
plt.ylabel('Average Path Length')

In [ ]:
shap.plots.bar(shap_values_tree)

In [ ]:
shap.plots.beeswarm(shap_values_tree)

In [ ]:
# Interaction values
shap_int_values=tree_explainer.shap_interaction_values(features)

In [ ]:
mean_shap=np.abs(shap_int_values).mean(0)
mean_shap=np.round(mean_shap, 1)

df=pd.DataFrame(mean_shap, index=features.columns, columns=features.columns)

df.where(df.values==np.diagonal(df), df.values*2, inplace=True)

sns.set(font_scale=1)
sns.heatmap(df, cmap='coolwarm', annot=True)
plt.yticks(rotation=0)